<font size="6">Модели на основе энкодера Трансформера</font>

Архитектура классического Трансформера состоит из энкодера и декодера. Она используется для задачи машинного перевода — преобразования одной последовательности в другую, длина которых может не совпадать (sequence-to-sequence).

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/transformer.png" width="400"></center>

<center><em>Источник: <a href="https://arxiv.org/abs/1706.03762">Attention Is All You Need</a></em></center>

Однако блоки энкодера и декодера можно использовать по отдельности.
- Модели на основе декодера применяются для генерации текста и используют маскированное внимание (Generative Pre-trained Transformers, GPT)
- Модели на основе энкодера применяются для других задач: классификации одного или пары предложений, теггирования последовательности, поиска ответа на вопрос (Bidirectional Encoder Representations from Transformers, BERT)

Сегодня мы подробно рассмотрим архитектуру модели BERT и её применение для различных задач. BERT возник как результат исправления недочетов предыдущих моделей, поэтому рассказ про него мы начнем немного издалека.

## Первая модель на улице Сезам — ELMo

Модель ELMo была представлена в статье [Deep contextualized word representations](https://arxiv.org/abs/1802.05365).

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/elmo.png" width="600"></center>

<center><em>Источник: <a href="https://arxiv.org/abs/1802.05365">Deep contextualized word representations</a></em></center>

Различным значениям слова *play* соответствуют разные контексты
употребления. Нужно передавать не только значение слова, но и контекстуальную
информацию – **контекстуализированные векторные
представления слов**. Контекстуализированные эмбеддинги присваивают словам разные векторы на основе их семантики в контексте предложения. Такие контекстуализированные векторы вычисляются посредством обучения языковой модели: Embeddings from Language Models = ELMo.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/elmo_2.png" width="800"></center>

<center><em>Источник: <a href="https://www.google.com/url?sa=i&url=https%3A%2F%2Fx.com%2Fcatherinehyeo%2Fstatus%2F1283883310519705600%3Flang%3Dar-x-fm&psig=AOvVaw1P-Ip7QomOk7i0co0sy1kV&ust=1735482361749000&source=images&cd=vfe&opi=89978449&ved=0CBcQjhxqFwoTCPCFh-DVyooDFQAAAAAdAAAAABAE">This trend started with ELMo</a></em></center>

Для обучения векторов ELMo используется двунаправленная языковая модель (bidirectional Language Model или biLM).

Модель вычисляет вероятность последовательности $t_1, t_2, \dots, t_N$.
Два прохода по тексту:
- прямой (forward):
  - информация об определенном слове и контексте перед ним
  - вероятность $t_k$ при условии предшествующего контекста $t_1, ..., t_{k-1}$
- обратный (backward):
  - информация о слове и контексте после него
  - вероятность $t_k$ при условии последующего контекста $t_{k+1}, \dots, t_N$

Важно: ELMo не имеет отношения к Трансформерам.

Модель состоит из двух слоев. На каждом слое обучается двунаправленная модель долгой краткосрочной памяти (biLSTM).

Информация из прямого и обратного прохода на первом слое формирует промежуточные векторы слов, которые подаются на вход второго слоя модели. Результирующие векторы — взвешенная сумма необработанных векторов и двух промежуточных векторов.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/elmo_architecture.png" width="600"></center>

<center><em>Источник: <a href="https://www.researchgate.net/publication/359157231_Identifying_Contradictions_in_the_Legal_Proceedings_Using_Natural_Language_Models">Identifying Contradictions in the Legal Proceedings Using Natural Language Models</a></em></center>

Поскольку модель обучается на задаче языкового моделирования, размеченные тексты не нужны, появляется возможность использовать большой объем данных для обучения. Модель выучивает некоторые общие знания о языке, не затачиваясь ни под какую конкретную задачу.

ELMo стала важным шагом к распространению переноса обучения в области NLP. Выходы модели ELMo могут использоваться как контекстуализированные эмбеддинги для различных задач обработки текста.

В случае word2vec каждому слову соответствует конкретный вектор, они могут быть сохранены в файл и затем взязы оттуда.

ELMo строит контекстно зависимые вектора. Чтобы получить вектор для слов предложения, нужно сначала пропустить это предложение через модель. Обучения уже не происходит.

📌 Фиксированы ли векторы ELMo для слова и его значений?

Векторы для слова *bank*: значения 'берег' и 'финансовая организация'.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bank_embeddings.png" width="600"></center>

<center><em>Источник: <a href="https://www.google.com/url?sa=i&url=https%3A%2F%2Ftowardsdatascience.com%2Fvisualizing-elmo-contextual-vectors-94168768fdaa&psig=AOvVaw2Fl8nHlIT3AsSv7fABstha&ust=1735482707884000&source=images&cd=vfe&opi=89978449&ved=0CBcQjhxqFwoTCOCok4TXyooDFQAAAAAdAAAAABAE">Visualizing ELMo Contextual Vectors</a></em></center>

## Вторая модель на улице Сезам — BERT

Модель BERT была представлена в статье [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805).

Идеи, которые были предложены ранее и удачно объединились при создании модели BERT:
- обучение на задаче языкового моделирования, которая не требует разметки, и перенос обучения (или предобучение) — ELMo
- механизм множественного внутреннего внимания, используемый без RNN, — Трансформер
- декодер Трансформера, который использует только левый контекст входного предложения, — GPT

Недостаток ELMo: анализируя левый и правый контекст отдельно с помощью biLSTM, мы можем терять часть информации. Хотелось бы учитывать левый и правый контекст одновременно.

Новшество BERT — использование **энкодера** Трансформера, чтобы получить "обогащенные" вниманием векторы слов.


<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bert.png" width="600"></center>

<center><em>Источник: <a href="https://www.linkedin.com/posts/robert-mcmenemy-%F0%9F%91%BE-70a0709b_efficient-language-modelling-with-bert-leveraging-activity-7247876446411452417-Fqzk">Efficient Language Modeling with BERT</a></em></center>

BERT состоит из нескольких последовательно соединенных блоков энкодера трансформера.

На вход модель получает последовательность токенов, на выходе отдает векторное представление для каждого токена, обогащенное контекстом. Энкодер содержит механизм внутреннего внимания (Self-Attention), который применяется к каждому токену и позволяет улавливать контекст.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bert_input.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

Две конфигурации:
- базовая (base): 12 слоев, размер скрытого слоя — 768, 110 миллионов весов
- расширенная (large): 24 слоя, размер скрытого слоя — 1024, 340 миллионов весов

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bert_base_large.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

BERT обучается на двух задачах:
- маскированное языковое моделирование (masked language modeling, MLM)
- предсказание следующего предложения (next sentence prediction, NSP)

### Маскированное языковое моделирование

15% случайно выбранных токенов по всему корпусу маскируется — заменяется на спецтокен [MASK]. Задача модели — предсказать наиболее вероятный токен на месте маски.

📌 Как можно получить вероятности слов из векторов на выходе энкодера?

Если модель всегда должна будет предсказывать наиболее вероятное слово только для масок, то для остальных слов векторы будут обучаться хуже. Нужно "обмануть" модель, чтобы она смотрела на все слова входной последовательности.

Среди выбранных 15% токенов:
- 80% маскируются: my dog is [MASK]
- 10% меняются на случайное слово: my dog is apple
- 10% остаются: my dog is hairy

Это разбиение меняется на каждой эпохе обучения.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/masked_language_modeling.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

📌 На что похожа задача маскированного языкового моделирования — предсказания слова по контексту?

Обучение на задаче маскированного языкового моделирования позволяет получить контекстуализированные векторы токенов. Предобученные векторы можно использовать в других задачах обработки текста.

### Предсказание следующего предложения

Помимо векторов слов, хотелось бы получать также векторные представления предложений. Для этого попробуем предсказать, следует ли одно предложение за другим.

Предложения разделены спецтокеном [SEP]. За классификацию отвечает спецтокен [CLS]. Он содержит представление обо всем предложении. Выход [CLS] токена пропускается через линейный слой размера 2.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/next_sentence_prediction.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

Положительные примеры представляют собой предложения, которые действительно следуют друг за другом в корпусе.

Вход: [CLS] the man went to the [MASK] store [SEP] he bought a gallon [MASK] milk [SEP]

Метка: IsNext

Отрицательные примеры представляют собой предложения, которые выбираются случайно.

Вход: [CLS] the man [MASK] to the store [SEP] penguin [MASK] are flight ##less birds [SEP]

Метка: NotNext

📌 Насколько хорошим является такой способ подбора отрицательных примеров?

Обучение происходит по двум задачам параллельно. Значение функции потерь считается отдельно для маскированного языкового моделирования по токену [MASK] и для предсказания следующего слова по токену [CLS].

## Библиотека Transformers

Библиотека [Transformers 🛠️[doc]](https://huggingface.co/docs/transformers/index) создана сообществом HuggingFace — это платформа и сообщество для разработки и обмена моделями машинного обучения в области обработки естественного языка (и не только). Здесь можно найти готовые модели, узнать об их параметрах и применении, а также делиться своими разработками и идеями с другими специалистами. Библиотека [Transformers 🛠️[doc]](https://huggingface.co/docs/transformers/index) позволяет работать с открытыми трансформерными моделями.

In [ ]:
!pip install transformers -q

В библиотеке реализованы классы для различных архитектур — в том числе, для модели [BERT 🛠️[doc]](https://huggingface.co/docs/transformers/model_doc/bert).

Нам понадобятся классы [BertTokenizer 🛠️[doc]](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertTokenizer) и [BertForMaskedLM 🛠️[doc]](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertForMaskedLM). Для загрузки конкретных моделей используется метод `.from_pretrained`.

BERT — это общее название архитектуры. С её использованием были обучены модели на различных языках и датасетах. Все модели, которые доступны на HuggingFace, можно посмотреть в разделе [Models 🛠️[doc]](https://huggingface.co/models). Чтобы загрузить модель, нужно указать её идентификатор.

### BertTokenizer

Загрузим токенизатор для модели [BERT base cased 🛠️[doc]](https://huggingface.co/bert-base-cased) для английского языка.

In [ ]:
from transformers import BertTokenizer
en_tz = BertTokenizer.from_pretrained("google-bert/bert-base-cased")

Токенизируем английское предложение с помощью метода `.tokenize()`.

In [ ]:
sent = "He remains characteristically confident and optimistic."
tokenized_sent = en_tz.tokenize(sent)
tokenized_sent

Если какое-то слово не представлено в словаре целиком, при токенизации оно делится на подслова.

Посмотрим, какие индексы в словаре соответствуют словам, с помощью метода `convert_tokens_to_ids()`.

In [ ]:
en_tz.convert_tokens_to_ids(tokenized_sent)

Загрузим токенизатор для модели на основе архитектуры BERT для другого языка (не английского) и попробуем подобрать предложение, где при токенизации одно или более слов делятся на подслова.

Пример для русского языка:

In [ ]:
ru_tz = BertTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

In [ ]:
ru_tz.tokenize("Сельскохозяйственно-машиностроительный"), ru_tz.tokenize("Частнопредпринимательский")

Пример для немецкого языка:

In [ ]:
de_tz = BertTokenizer.from_pretrained("google-bert/bert-base-german-cased")

In [ ]:
de_tz.tokenize("die Aufmerksamkeitsdefizitstörung"), de_tz.tokenize("das Ampelmännchen")

### BertForMaskedLM

Загрузим саму модель [BERT base cased 🛠️[doc]](https://huggingface.co/bert-base-cased) для английского языка.

In [ ]:
from transformers import BertForMaskedLM
en_model = BertForMaskedLM.from_pretrained("google-bert/bert-base-cased")

Поскольку модель обучалась на задаче маскированного языкового моделирования, она способна предсказывать наиболее вероятные слова на месте спецтокена [MASK].

Напишем функцию `predict_mask`, которая находит распределение вероятностей для маски.
- Добавим спецтокены [CLS] и [SEP]
- Токенизируем текст (`text`)
- Определим индекс маскированного слова
- Переведем токенизированные слова (`tokenized_text`) в индексы
- Запишем индексы в тензор
- Применим модель к токенизированному предложению
- Запишем выходы модели для каждого слова
- Применим софтмакс (`torch.softmax()`) к результатам для маскированного слова, его найдем среди всех выходов модели (`predictions`) по индексу (`masked_index`)
- Запишем k самых больших значений весов и их индексы, которые соответствуют словам в словаре
- Пройдем в цикле по списку индексов
  - Переведем каждый индекс в соответствующий токен
  - Запишем его вероятность

In [ ]:
import torch

def predict_mask(tokenizer, model, text, top_k=5):

    text = f"[CLS] {text} [SEP]"
    tokenized_text = tokenizer.tokenize(text)
    masked_index = tokenized_text.index("[MASK]")
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])

    with torch.no_grad():
        outputs = model(tokens_tensor)
        predictions = outputs["logits"].squeeze() # token_len x vocabulary_size

    probs = torch.softmax(predictions[masked_index,:], dim=-1)
    top_k_weights, top_k_indices = torch.topk(probs, top_k)

    for i, pred_idx in enumerate(top_k_indices):
        predicted_token = tokenizer.convert_ids_to_tokens([pred_idx])[0]
        token_weight = top_k_weights[i]
        print("[MASK]: '%s'"%predicted_token, " | weights:", float(token_weight))

In [ ]:
predict_mask(en_tz, en_model, "My [MASK] is so cute.", top_k=5)

Загрузим модель на основе архитектуры BERT для другого языка (не английского) и применим функцию к предложению, где одно слово маскировано.

Пример для русского языка:

In [ ]:
ru_model = BertForMaskedLM.from_pretrained("DeepPavlov/rubert-base-cased")

In [ ]:
predict_mask(ru_tz, ru_model, "Моя [MASK] очень милая.", top_k=5)

Пример для английского языка:

In [ ]:
de_model = BertForMaskedLM.from_pretrained("google-bert/bert-base-german-cased")

In [ ]:
predict_mask(de_tz, de_model, "Meine [MASK] ist sehr nett.", top_k=5)